# Module 13: Interactive FastAPI & Modern ASGI Architecture

### What You Will Discover
By running this notebook, you will explore the raw ASGI protocol triple `(scope, receive, send)`, define typed FastAPI routes, and issue asynchronous test requests using `httpx.AsyncClient`.

**Key Question Answered:** *How does FastAPI automatically parse, validate, and document HTTP endpoints at C-speed without manual parsing code?*


In [ ]:
# Step 1: Defining a minimal FastAPI application
from fastapi import FastAPI, status
from pydantic import BaseModel

app = FastAPI(title='Interactive Diagnostic API')

class Item(BaseModel):
    name: str
    price: float


In [ ]:
# Step 2: Registering route handlers
@app.get('/health')
async def health_check():
    return {'status': 'healthy', 'uptime': '99.99%'}

@app.post('/items', status_code=status.HTTP_201_CREATED)
async def create_item(item: Item):
    return {'item_name': item.name, 'total_with_tax': item.price * 1.08}


In [ ]:
# Step 3: Executing in-memory async requests with httpx TestClient / AsyncClient
from httpx import ASGITransport, AsyncClient

transport = ASGITransport(app=app)
async with AsyncClient(transport=transport, base_url='http://test') as client:
    res = await client.get('/health')
    print(f'GET /health -> Status: {res.status_code}, Body: {res.json()}')


### 🔮 Prediction Prompt
**Before running the next cell:** If we send `POST /items` with invalid JSON `{"name": "Widget", "price": "not_a_number"}`, what HTTP status code will FastAPI return? Will it crash the server?


In [ ]:
# Surprising Result: Automatic 422 Unprocessable Entity
async with AsyncClient(transport=transport, base_url='http://test') as client:
    bad_res = await client.post('/items', json={'name': 'Widget', 'price': 'not_a_number'})
    print(f'Status: {bad_res.status_code}')
    print(f'Error Payload: {bad_res.json()}')
    print('Explanation: Pydantic rejects invalid types automatically with exact JSON path locations!')


### Raw ASGI App: The Underlying Mechanism
FastAPI and Starlette are built on top of the ASGI specification. Here is a raw ASGI callable.


In [ ]:
async def minimal_asgi(scope, receive, send):
    assert scope['type'] == 'http'
    await send({'type': 'http.response.start', 'status': 200, 'headers': [[b'content-type', b'text/plain']]})
    await send({'type': 'http.response.body', 'body': b'Raw ASGI Response'})

raw_transport = ASGITransport(app=minimal_asgi)
async with AsyncClient(transport=raw_transport, base_url='http://test') as client:
    raw_res = await client.get('/')
    print(f'Raw ASGI status: {raw_res.status_code}, text: {raw_res.text}')


### Inspecting Automatic OpenAPI JSON Documentation
FastAPI compiles Pydantic models and routes into OpenAPI 3.1 JSON schemas automatically.


In [ ]:
openapi_schema = app.openapi()
print(f'OpenAPI Title: {openapi_schema["info"]["title"]}')
print(f'Endpoints declared: {list(openapi_schema["paths"].keys())}')


### 🛠️ Interactive Challenge: Fix the Blocking Route
The following route handler attempts to process data but uses synchronous `time.sleep()`, blocking all other concurrent requests on the event loop. Fix it to use `await asyncio.sleep()` or `asyncio.to_thread`.


In [ ]:
# TODO: FIX ME - Replace time.sleep() with non-blocking await asyncio.sleep()
import asyncio


@app.get('/delay')
async def non_blocking_delay():
    # FIX: await asyncio.sleep(0.05)
    await asyncio.sleep(0.05)
    return {'status': 'resumed'}

async with AsyncClient(transport=transport, base_url='http://test') as client:
    res = await client.get('/delay')
    print(f'Non-blocking delay response: {res.json()}')


### 🏁 Summary & Next Steps
- ASGI standardizes `(scope, receive, send)` asynchronous interfaces.
- FastAPI couples Pydantic validation with Starlette routing.
- In-memory testing with `httpx.ASGITransport` avoids network port bindings.
- Run `python 01_asgi_raw_interface_demo.py` and `python 02_fastapi_routing_demo.py`.
- Follow [PROJECT_GUIDE.md](PROJECT_GUIDE.md) to implement the ingestion microservice.
